# STAGE C/5 — Fine-tune NER (XLM-R large) từ DAPT + silver **(v1 = 34.60)**

**Add Input (BẮT BUỘC):** output **Stage A** (dapt) + output **Stage B** (silver, llm).
**Settings:** GPU **T4 (1 con)** + Internet ON. **~30–40′**. Xong → **Commit** cho Stage E.

Encoder học từ silver (miền thật) + llm + template. *(Bản v2 distant/khử-nhiễu đo trên
leaderboard XẤU hơn → tạm bỏ; muốn A/B thì chạy `scripts/prep_v2.py` thủ công.)*


In [ ]:
# Clone repo
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: tránh DataParallel chậm
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
!pip install -q rapidfuzz pyyaml regex accelerate sentencepiece


In [ ]:
# Tìm artifact từ stage trước (đã Add Input)
import glob, os, shutil
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
def find_model(name):
    # model dir = thư mục có config.json (TRÁNH khớp nhầm src/ner là code)
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}/config.json", recursive=True)
            if "/src/" not in h]
    return os.path.dirname(hits[0]) if hits else None


In [ ]:
DAPT = find_model('dapt') or 'xlm-roberta-large'
SILVER = find_input('silver.jsonl')
LLM = find_input('llm.jsonl')
print('DAPT:', DAPT); print('SILVER:', SILVER); print('LLM:', LLM)
if DAPT == 'xlm-roberta-large': print('!! chua Add Input Stage A -> mat DAPT')
if not SILVER: print('!! chua Add Input Stage B -> thieu silver')


In [ ]:
# template (sàn, offset chuẩn)
!python scripts/gen_synth.py --n_train 4000 --n_dev 400


In [ ]:
# gộp nguồn có thật rồi fine-tune (large + adafactor + grad-ckpt = vừa T4 15GB)
parts = [p for p in [SILVER, LLM, 'data/synthetic/train.jsonl'] if p]
train_arg = ','.join(parts)
print('TRAIN:', train_arg)
!python scripts/train_ner.py --model {DAPT} --train {train_arg} \
   --epochs 3 --batch_size 8 --grad_accum 2 --max_length 384 --optim adafactor \
   --out models/ner


In [ ]:
# Lưu model ra /kaggle/working để Commit
import shutil, os
dst = "/kaggle/working/ner"
if os.path.isdir("models/ner"):
    shutil.rmtree(dst, ignore_errors=True); shutil.copytree("models/ner", dst)
    print("SAVED ->", dst, "| Commit roi qua Stage E")
else:
    print("!! train chưa tạo models/ner")


**OOM?** `--batch_size 4 --grad_accum 4`. **Quên Add Input A** → DAPT = 'xlm-roberta-large' (mất DAPT).
